# Tables (TableView)

**Part I · Visualization** — Tutorial 17

Edit tabular data with the native, dependency-free `TableView` grid. You will
learn to:

- Define columns, rows, and typed columns (`number`/`string`/`bool`/enum).
- Edit cells, sort, and navigate with the keyboard.
- Drive rows and columns from the backend (`add_row`/`add_column`/`insert_row`/
  `insert_column`/`delete_row`/`delete_column`).
- Undo/redo edits and react to bulk changes (`on_change`).
- Persist the grid as JSON (`save`/`load`) or CSV (`to_csv`/`from_csv`).

> `TableView` replaces the old `add_table` facade. It is a plain-DOM grid themed
> by the `--tanga-*` tokens, so it styles like every other control.

> **No graphical output in these docs.** These cells build the app but don't run it,
> so no image is rendered here. Run the notebook in Jupyter and call the app's
> `.run()` to open the viewer in a browser.


## Setup


In [1]:
import tempfile
from pathlib import Path

from pytanga.viz import TableView

tmp = Path(tempfile.mkdtemp())  # scratch dir for the persistence examples


## 1. A basic table

`TableView` takes `columns` (headers) and `rows` (data). `on_cell_change` fires
with a `TableCellChange` (`.row`, `.col`, `.value`) on every edit.


In [2]:
async def on_cell_change(change, _event):
    print(f"cell ({change.row}, {change.col}) = {change.value!r}")

table = TableView(
    "data",
    label="Data",
    columns=["x", "y", "active", "status"],
    rows=[[1.5, 2.5, True, "on"], [3.5, 4.5, False, "off"]],
    on_cell_change=on_cell_change,
)

print("columns:", table.columns)
print("rows:", table.rows)


columns: ['x', 'y', 'active', 'status']
rows: [[1.5, 2.5, True, 'on'], [3.5, 4.5, False, 'off']]


## 2. Column types

`column_types` sets one entry per column: `"number"` (right-aligned, rejects
non-numeric input), `"string"`, `"bool"` (an always-on checkbox), or a list of
allowed values (an enum dropdown). Omitted entries deduce from the data.


In [3]:
table = TableView(
    "data",
    label="Data",
    columns=["x", "y", "name", "active", "status"],
    rows=[
        [1.5, 2.5, "alpha", True, "on"],
        [3.5, 4.5, "beta", False, "off"],
    ],
    column_types=["number", "number", "string", "bool", ["on", "off"]],
)

# A `number` column carries a Python str.format display template.
table.set_column_format(0, "{:.2f} m")
print("typed columns set; x is formatted as '{:.2f} m'")


typed columns set; x is formatted as '{:.2f} m'


## 3. Backend-driven rows and columns

Mutate the model and push the grid from the backend: `add_row`/`add_column`
append, `insert_row(index)`/`insert_column(index)` insert relative to the
selected cell (falling back to the edges), and `delete_row`/`delete_column`
remove a row/column.


In [4]:
table = TableView(
    "data",
    label="Data",
    columns=["x", "y"],
    rows=[[1.0, 2.0]],
    column_types=["number", "number"],
)

table.add_row()          # append an empty row
print("after add_row:", table.rows)

table.add_column("z")    # append a column
print("after add_column:", table.columns, table.rows)

table.insert_row(0)      # insert an empty row at the top
print("after insert_row(0):", table.rows)

table.delete_row(1)      # delete the second row
print("after delete_row(1):", table.rows)


after add_row: [[1.0, 2.0], ['', '']]
after add_column: ['x', 'y', 'z'] [[1.0, 2.0, ''], ['', '', '']]
after insert_row(0): [['', '', ''], [1.0, 2.0, ''], ['', '', '']]
after delete_row(1): [['', '', ''], ['', '', '']]


## 4. Undo/redo

Undo/redo history is kept backend-side (Ctrl+Z / Ctrl+Shift+Z / Ctrl+Y in the
browser, or `undo()`/`redo()`). `clear_history()` resets it.


In [5]:
table.add_row()
print("can_undo:", table.can_undo)

table.undo()
print("after undo:", table.rows)

table.redo()
print("after redo:", table.rows)

table.clear_history()
print("after clear_history, can_undo:", table.can_undo)


can_undo: True
after undo: [['', '', ''], ['', '', '']]
after redo: [['', '', ''], ['', '', ''], ['', '', '']]
after clear_history, can_undo: False


## 5. Persistence: JSON and CSV

`save(path)`/`load(path)` round-trip data + types + column widths + row height +
sort as a versioned JSON file. `to_csv(path)`/`from_csv(path)` exchange plain
data; import auto-detects the delimiter (`;` vs `,`) and decimal separator (`,`
vs `.`), and both can be overridden with `delimiter=` / `decimal_separator=`.


In [6]:
table.save(str(tmp / "table.json"))

table.to_csv(str(tmp / "table.csv"))
print("saved JSON + CSV to", tmp)

# Round-trip: load from CSV with a custom dialect, then reload the JSON.
loaded = TableView("loaded", label="Loaded", columns=["x", "y", "z"])
loaded.from_csv(str(tmp / "table.csv"))
print("loaded from CSV:", loaded.columns, loaded.rows)

loaded.load(str(tmp / "table.json"))
print("loaded from JSON:", loaded.columns, loaded.rows)


saved JSON + CSV to /tmp/tmpg0pag47y
loaded from CSV: ['x', 'y', 'z'] [['', '', ''], ['', '', ''], ['', '', '']]
loaded from JSON: ['x', 'y', 'z'] [['', '', ''], ['', '', ''], ['', '', '']]


## 6. Enum columns: `column` and `custom`

Two dynamic enum column types sit beside the fixed enum:

- `column` — a dropdown of the de-duped values of **another column**
  (`{"kind": "column", "source": <index>}`).
- `custom` — a backend-only enum whose dropdown is populated at edit time by an
  `on_enum_options` handler.


In [7]:
seating = TableView(
    "seating",
    label="Seating",
    columns=["name", "preference"],
    rows=[["Alice", "window"], ["Bob", "aisle"]],
    column_types=["string", {"kind": "column", "source": 0}],
)
print("column-fed enum table created")

custom = TableView(
    "custom",
    label="Custom",
    columns=["key"],
    rows=[["v1"]],
    column_types=["custom"],  # dropdown populated by on_enum_options
)
print("custom enum table created")


column-fed enum table created
custom enum table created


## Visual Examples

An editable table mounted in a layout with backend-driven buttons (condensed
from the library's `table_data.py`).


In [8]:
from pytanga.geometry import Point, Sphere
from pytanga.viz import ButtonView, GroupView, SceneView, VisualizerApp


class TableApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="Table Data")
        self._columns = ["x", "y", "active", "status"]
        self._rows = [[1.5, 2.5, True, "on"], [3.5, 4.5, False, "off"]]
        self._table = None

    async def init(self) -> None:
        self.viz.add(Sphere(Point(0, 0, 0), 1.0), entity_id="ball",
                     color="#4488ff", opacity=0.9)
        self._table = TableView(
            "data",
            label="Data",
            columns=self._columns,
            rows=self._rows,
            column_types=[None, None, "bool", ["on", "off"]],
            on_cell_change=self.on_cell_change,
            on_row_add=self.on_row_add,
            on_column_add=self.on_column_add,
        )
        self.viz.set_layout(
            SceneView(
                "",
                overlay=[
                    GroupView(
                        "",
                        [
                            self._table,
                            ButtonView("reset", label="Reset table", on_click=self.on_reset),
                            ButtonView("add_row", label="+ Row", on_click=self.on_add_row),
                            ButtonView("add_column", label="+ Column", on_click=self.on_add_column),
                        ],
                        position="bottom-right",
                    )
                ],
            )
        )
        self.viz.flush()

    async def on_cell_change(self, change, _event):
        self.viz.set_annotation(f"Cell ({change.row}, {change.col}) = {change.value!r}")

    async def on_row_add(self, add, _event):
        self.viz.set_annotation(f"Added row {add.row}")

    async def on_column_add(self, add, _event):
        self.viz.set_annotation(f"Added column {add.col} ({add.header!r})")

    async def on_reset(self, _value, _event):
        self._table.set_value({"columns": self._columns, "rows": self._rows})

    async def on_add_row(self, _value, _event):
        self._table.add_row()

    async def on_add_column(self, _value, _event):
        self._table.add_column(f"C{len(self._table.columns) + 1}")


# TableApp().run()
print("TableApp defined")


TableApp defined


## Summary

| Task | API |
|---|---|
| Construct | `TableView(cid, columns=..., rows=..., column_types=..., ...)` |
| Edit | double-click a cell (typed columns validate input) |
| Sort | click a column header (`sortable=True`) |
| Navigate | Tab / Shift+Tab / Enter / arrow keys (active cell) |
| Backend rows/columns | `add_row` / `add_column` / `insert_row` / `insert_column` / `delete_row` / `delete_column` |
| Undo/redo | Ctrl+Z / Ctrl+Shift+Z / Ctrl+Y · `undo()` / `redo()` / `clear_history()` |
| Bulk change | `on_change` (fires with the full grid on undo/redo) |
| Persistence | `save` / `load` (JSON) · `to_csv` / `from_csv` (delimiter/decimal) |
| Enum columns | `column` (from another column) · `custom` (`on_enum_options`) |

**Next:** [18 — Banners & Dialogs](../18_banners_dialogs/).
